# 01 · Preparación de datos para el modelado

## 1. Carga de datos

In [1]:
import polars as pl

In [2]:
# Carga lazy con scan_parquet() para optimizar el rendimiento en datasets grandes.
fact = pl.scan_parquet("../data/gold/fact_trafico_hora.parquet")
fecha = pl.scan_parquet("../data/gold/dim_fecha.parquet")
sensor = pl.scan_parquet("../data/gold/dim_sensor.parquet")

In [3]:
print(f"Fact: ({fact.select(pl.len()).collect().item()}, {len(fact.collect_schema().names())})")
print(f"Fecha: ({fecha.select(pl.len()).collect().item()}, {len(fecha.collect_schema().names())})")
print(f"Sensor: ({sensor.select(pl.len()).collect().item()}, {len(sensor.collect_schema().names())})")

Fact: (40521393, 13)
Fecha: (365, 8)
Sensor: (5088, 9)


## 2. Exploración inicial

In [4]:
fact.collect_schema()

Schema([('id_sensor', Int32),
        ('id_fecha', Date),
        ('hora', Int32),
        ('intensidad_media', Float64),
        ('intensidad_max', Float64),
        ('intensidad_min', Float64),
        ('ocupacion_media', Float64),
        ('ocupacion_max', Float64),
        ('velocidad_media', Float64),
        ('velocidad_min', Float64),
        ('num_mediciones', Int64),
        ('num_error_E', Float64),
        ('porcentaje_calidad', Float64)])

In [5]:
fecha.collect_schema()

Schema([('id_fecha', Date),
        ('año', Int64),
        ('mes', Int64),
        ('nombre_mes', String),
        ('trimestre', Int64),
        ('dia', Int64),
        ('dia_semana', Int64),
        ('fin_semana', Boolean)])

In [6]:
sensor.collect_schema()

Schema([('id_sensor', Int32),
        ('tipo_elem', String),
        ('distrito', Int32),
        ('cod_cent', String),
        ('nombre_norm', String),
        ('utm_x', Float64),
        ('utm_y', Float64),
        ('latitud', Float64),
        ('longitud', Float64)])

In [7]:
fact.head().collect()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,ocupacion_media,ocupacion_max,velocidad_media,velocidad_min,num_mediciones,num_error_E,porcentaje_calidad
i32,date,i32,f64,f64,f64,f64,f64,f64,f64,i64,f64,f64
1001,2026-04-20,15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-22,17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-23,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-26,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0
1001,2026-04-27,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0.0,100.0


In [8]:
fecha.head().collect()

id_fecha,año,mes,nombre_mes,trimestre,dia,dia_semana,fin_semana
date,i64,i64,str,i64,i64,i64,bool
2026-04-20,2026,4,"""April""",2,20,1,false
2026-04-26,2026,4,"""April""",2,26,0,true
2026-04-19,2026,4,"""April""",2,19,0,true
2026-04-15,2026,4,"""April""",2,15,3,false
2026-04-28,2026,4,"""April""",2,28,2,false


In [9]:
sensor.head().collect()

id_sensor,tipo_elem,distrito,cod_cent,nombre_norm,utm_x,utm_y,latitud,longitud
i32,str,i32,str,str,f64,f64,f64,f64
1002,"""other""",10,"""05FT37PM01""","""05ft37pm01""",436892.118106,4.4733e6,40.40803,-3.74376
1009,"""other""",9,"""03FT52PM01""","""03ft52pm01""",438499.044504,4.4742e6,40.416234,-3.724909
1011,"""other""",9,"""18RM17PM01""","""18rm17pm01""",438656.420188,4.4744e6,40.418234,-3.723076
1012,"""other""",9,"""18RA66PM01""","""18ra66pm01""",438740.943152,4.4746e6,40.419861,-3.722097
1015,"""other""",9,"""18NC52PM02""","""18nc52pm02""",438763.057485,4.4747e6,40.420488,-3.721843


## 3. Validación antes del join

In [10]:
# Obtener el número de sensores únicos en cada tabla
sensores_fact = fact.select(pl.col("id_sensor").n_unique()).collect().item()
sensores_dim = sensor.select(pl.col("id_sensor").n_unique()).collect().item()

print(f"Sensores fact: {sensores_fact}")
print(f"Sensores dimensión: {sensores_dim}")

Sensores fact: 4933
Sensores dimensión: 5088


In [11]:
# Sensores presentes en la tabla de hechos y ausentes en la dimensión
sensores_fact_set = set(fact.select("id_sensor").unique().collect()["id_sensor"])
sensores_dim_set = set(sensor.select("id_sensor").unique().collect()["id_sensor"])

sensores_faltantes = sensores_fact_set - sensores_dim_set
print(len(sensores_faltantes))

0


## 4. Integración de tablas

In [12]:
df = (
    fact
    .join(fecha, on="id_fecha", how="left")
    .join(sensor, on="id_sensor", how="left")
)

## 5. Verificación de la integración

In [ ]:
# Verificar que el join no ha duplicado ni perdido registros.
print(f"Filas en fact: {fact.select(pl.len()).collect().item()}")
print(f"Filas en df: {df.select(pl.len()).collect().item()}")

Filas en fact: 40521393
Filas en df: 40521393


In [ ]:
# Verificar que todas las claves de fact encontraron correspondencia en las tablas dimensión.
df.select(
    pl.col("año").is_null().sum()
).collect()

año
u32
0


In [ ]:
# Verificar que todas las claves de fact encontraron correspondencia en las tablas dimensión.
df.select(
    pl.col("tipo_elem").is_null().sum()
).collect()

tipo_elem
u32
0


In [ ]:
# Verificar que el join no ha generado duplicados.
clave_unica = df.select(
    pl.struct(
        ["id_sensor", "id_fecha", "hora"]
    ).n_unique()
).collect().item()

total_filas = df.select(pl.len()).collect().item()

print(f"Filas: {total_filas}")
print(f"Claves únicas: {clave_unica}")

Filas: 40521393
Claves únicas: 40521393


## 6. Tratamiento de valores nulos

In [ ]:
(
    df.null_count()
      .collect() 
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""nombre_norm""",111090
"""distrito""",42529
"""velocidad_media""",33781
"""velocidad_min""",33781
"""id_sensor""",0
…,…
"""cod_cent""",0
"""utm_x""",0
"""utm_y""",0


In [18]:
df = df.drop(
    "nombre_norm",
    "cod_cent",
    "utm_x",
    "utm_y"
)

Se descartan variables que no aportan información relevante para la predicción o cuya información ya está representada por otras columnas del conjunto de datos. Esto permite reducir la dimensionalidad del dataset y facilitar las etapas posteriores de análisis y modelado.

In [19]:
df = df.with_columns(
    pl.col("distrito")
      .fill_null(0)
      .cast(pl.Int32)
)

Los valores nulos de la columna distrito se reemplazan por 0, utilizándolo como código para representar un distrito desconocido o no asignado. Posteriormente, la columna se convierte al tipo Int32 para mantener un formato numérico consistente.

In [20]:
df.group_by("tipo_elem").agg([
    pl.col("velocidad_media")
      .filter(pl.col("velocidad_media") > 0)
      .len()
      .alias("registros_con_velocidad")
]).collect()

tipo_elem,registros_con_velocidad
str,u32
"""M30""",2361374
"""URB""",1
"""other""",762297


In [21]:
# Tratamiento inicial de la velocidad

df = df.with_columns([

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_media") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_media"))
      .alias("velocidad_media"),

    pl.when(pl.col("tipo_elem") == "URB")
      .then(0)
      .when(pl.col("velocidad_min") < 0)
      .then(None)
      .otherwise(pl.col("velocidad_min"))
      .alias("velocidad_min")

])

Se normalizan las variables velocidad_media y velocidad_min aplicando dos reglas: para los elementos de tipo URB se asigna el valor 0, ya que la velocidad no resulta representativa, y los valores negativos se reemplazan por null al considerarse datos no válidos. El resto de los registros se mantienen sin cambios.

In [22]:
# Calcular la mediana por sensor

medianas = (
    df.group_by("id_sensor")
      .agg([
          pl.col("velocidad_media").median().alias("mediana_media"),
          pl.col("velocidad_min").median().alias("mediana_min")
      ])
)

df = df.join(medianas, on="id_sensor", how="left")

In [23]:
# Imputar los nulos con la mediana del sensor

df = df.with_columns([

    pl.col("velocidad_media")
      .fill_null(pl.col("mediana_media"))
      .alias("velocidad_media"),

    pl.col("velocidad_min")
      .fill_null(pl.col("mediana_min"))
      .alias("velocidad_min")

]).drop(["mediana_media", "mediana_min"])

In [24]:
df.select([
    pl.col("velocidad_media").is_null().sum(),
    pl.col("velocidad_min").is_null().sum()
]).collect()

velocidad_media,velocidad_min
u32,u32
0,0


In [25]:
(
    df.null_count()
      .collect()
      .transpose(
          include_header=True,
          header_name="columna",
          column_names=["nulos"]
      )
      .sort("nulos", descending=True)
)

columna,nulos
str,u32
"""id_sensor""",0
"""id_fecha""",0
"""hora""",0
"""intensidad_media""",0
"""intensidad_max""",0
…,…
"""fin_semana""",0
"""tipo_elem""",0
"""distrito""",0


## 7. Tratamiento de valores atípicos

In [27]:
# Comprobar si existen registros con el valor 99999
print(
    f"Registros con intensidad_max = 99999: "
    f"{df.filter(pl.col('intensidad_max') == 99999).select(pl.len()).collect().item()}"
)

Registros con intensidad_max = 99999: 1


In [29]:
# Inspeccionar el registro antes de eliminarlo
df.filter(
    pl.col("intensidad_max") == 99999
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
]).collect()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,num_mediciones
i32,date,i32,f64,f64,f64,i64
7112,2025-12-12,2,50083.0,99999.0,167.0,2


In [30]:
# Analizar las intensidades más elevadas del conjunto de datos
df.filter(
    pl.col("intensidad_media") > 10000
).select([
    "id_sensor",
    "id_fecha",
    "hora",
    "intensidad_media",
    "intensidad_max",
    "intensidad_min",
    "num_mediciones"
]).sort(
    "intensidad_media",
    descending=True
).collect()

id_sensor,id_fecha,hora,intensidad_media,intensidad_max,intensidad_min,num_mediciones
i32,date,i32,f64,f64,f64,i64
7023,2026-02-20,22,91368.333333,91544.0,91034.0,3
7023,2026-02-17,13,91338.25,91560.0,90707.0,4
7023,2026-02-20,23,91269.25,91517.0,90560.0,4
7023,2026-02-17,3,91191.5,91445.0,90932.0,4
7023,2026-02-21,7,91068.0,91500.0,90636.0,2
…,…,…,…,…,…,…
5670,2025-08-20,17,10283.0,22626.0,4889.0,4
5670,2025-07-16,10,10038.25,11225.0,8136.0,4
5670,2026-03-18,15,10033.75,13213.0,4523.0,4


In [31]:
# Eliminar únicamente el registro con el valor centinela 99999
df = df.filter(
    pl.col("intensidad_max") != 99999
)

Se elimina el único registro con intensidad_max = 99999, ya que este valor corresponde a un código centinela utilizado para representar un dato no válido y no una medición real de intensidad. Mantenerlo podría afectar negativamente al análisis y al entrenamiento del modelo.

## 8. Generación del dataset de modelado

In [32]:
df.sink_parquet("../data/modeling/base_modelado.parquet")